# {{TITLE}}

Compétition : [{{SLUG}}](https://www.kaggle.com/competitions/{{SLUG}}) · métrique : **{{METRIC}}**

Starter : EDA rapide → CV KFold (CatBoost / XGBoost) → Optuna sur sous-échantillon → fit sur tout le train → submission.
Soumettre ensuite depuis un terminal : `kg submit {{SLUG}} submissions/<fichier>.csv -m "message" --cv <score>`

In [ ]:
import os, warnings
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
warnings.filterwarnings("ignore")

DATA = Path("data")
SUB_DIR = Path("submissions"); SUB_DIR.mkdir(exist_ok=True)
SEED, N_FOLDS = 42, 5
N_JOBS = os.cpu_count()

train = pd.read_csv(DATA / "train.csv")
test = pd.read_csv(DATA / "test.csv")
sample = pd.read_csv(next(DATA.glob("*submission*.csv")))   # sample_submission.csv, gender_submission.csv…

ID = sample.columns[0]
TARGET = sample.columns[1]          # adapter si plusieurs cibles
print(train.shape, test.shape, "| id:", ID, "| target:", TARGET)
train.head()

## EDA rapide

In [ ]:
display(train.describe(include="all").T)
print("NaN train:\n", train.isna().sum()[lambda s: s > 0])
train[TARGET].hist(bins=50) if train[TARGET].nunique() > 20 else train[TARGET].value_counts().plot.bar()
plt.title(TARGET); plt.show()

## Features et type de tâche

In [ ]:
X = train.drop(columns=[ID, TARGET])
y = train[TARGET]
X_test = test.drop(columns=[ID])

cat_cols = X.select_dtypes(include=["object", "category", "bool"]).columns.tolist()
for c in cat_cols:
    X[c] = X[c].astype(str).astype("category")
    X_test[c] = pd.Categorical(X_test[c].astype(str), categories=X[c].cat.categories)

METRIC = "{{METRIC}}".lower()
IS_CLF = y.nunique() <= 20 or any(k in METRIC for k in ["auc", "accuracy", "logloss", "log loss", "f1", "map@"])
N_CLASSES = y.nunique() if IS_CLF else None
if IS_CLF and y.dtype == object:
    classes = np.sort(y.unique()); y = y.map({c: i for i, c in enumerate(classes)})
print("classification" if IS_CLF else "régression", "| catégorielles:", cat_cols)

In [ ]:
from sklearn.model_selection import KFold, StratifiedKFold
from sklearn.metrics import roc_auc_score, mean_squared_error, accuracy_score, log_loss

def score(y_true, p):
    if "auc" in METRIC:
        return roc_auc_score(y_true, p, multi_class="ovr") if p.ndim > 1 else roc_auc_score(y_true, p)
    if "rmsle" in METRIC:
        return np.sqrt(mean_squared_error(np.log1p(y_true), np.log1p(np.clip(p, 0, None))))
    if "accuracy" in METRIC:
        return accuracy_score(y_true, p.argmax(1) if p.ndim > 1 else (p > .5).astype(int))
    if "log" in METRIC:
        return log_loss(y_true, p)
    return np.sqrt(mean_squared_error(y_true, p))   # RMSE par défaut

HIGHER_IS_BETTER = any(k in METRIC for k in ["auc", "accuracy", "f1", "map"])
folds = (StratifiedKFold if IS_CLF else KFold)(N_FOLDS, shuffle=True, random_state=SEED)
print("métrique:", METRIC, "| plus haut = mieux" if HIGHER_IS_BETTER else "| plus bas = mieux")

## Modèles et CV

In [ ]:
from catboost import CatBoostClassifier, CatBoostRegressor
from xgboost import XGBClassifier, XGBRegressor

def make_model(kind, params):
    if kind == "cat":
        base = dict(iterations=2000, learning_rate=0.05, depth=6, random_seed=SEED, verbose=0,
                    thread_count=N_JOBS, early_stopping_rounds=100, cat_features=cat_cols)
        base.update(params)
        return (CatBoostClassifier if IS_CLF else CatBoostRegressor)(**base)
    base = dict(n_estimators=2000, learning_rate=0.05, max_depth=6, subsample=0.8, colsample_bytree=0.8,
                tree_method="hist", enable_categorical=True, random_state=SEED, n_jobs=N_JOBS,
                early_stopping_rounds=100)
    base.update(params)
    return (XGBClassifier if IS_CLF else XGBRegressor)(**base)

def predict(m, X_):
    if not IS_CLF:
        return m.predict(X_)
    p = m.predict_proba(X_)
    return p[:, 1] if N_CLASSES == 2 else p

def run_cv(kind, params, X_=None, y_=None, verbose=True):
    X_ = X if X_ is None else X_; y_ = y if y_ is None else y_
    oof = np.zeros((len(X_), N_CLASSES) if IS_CLF and N_CLASSES > 2 else len(X_))
    iters = []
    for k, (tr, va) in enumerate(folds.split(X_, y_)):
        m = make_model(kind, params)
        m.fit(X_.iloc[tr], y_.iloc[tr], eval_set=[(X_.iloc[va], y_.iloc[va])],
              **({} if kind == "cat" else {"verbose": False}))
        oof[va] = predict(m, X_.iloc[va])
        iters.append(m.get_best_iteration() if kind == "cat" else m.best_iteration)
        if verbose: print(f"  fold {k}: {score(y_.iloc[va], oof[va]):.5f}")
    s = score(y_, oof)
    if verbose: print(f"{kind} CV = {s:.5f}  (best iters ~{int(np.mean(iters))})")
    return s, oof, int(np.mean(iters))

baseline = {k: run_cv(k, {}) for k in ["cat", "xgb"]}

## Optuna (sur sous-échantillon pour aller vite)

In [ ]:
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

MODEL = "cat"          # "cat" ou "xgb"
N_TRIALS = 30
SUBSAMPLE = min(len(X), 100_000)
idx = X.sample(SUBSAMPLE, random_state=SEED).index
Xs, ys = X.loc[idx].reset_index(drop=True), y.loc[idx].reset_index(drop=True)

def objective(trial):
    if MODEL == "cat":
        p = dict(learning_rate=trial.suggest_float("learning_rate", 0.02, 0.2, log=True),
                 depth=trial.suggest_int("depth", 4, 9),
                 l2_leaf_reg=trial.suggest_float("l2_leaf_reg", 1, 20, log=True),
                 random_strength=trial.suggest_float("random_strength", 0, 2),
                 bootstrap_type="Bernoulli", subsample=trial.suggest_float("subsample", 0.6, 1.0))
    else:
        p = dict(learning_rate=trial.suggest_float("learning_rate", 0.02, 0.2, log=True),
                 max_depth=trial.suggest_int("max_depth", 3, 10),
                 min_child_weight=trial.suggest_float("min_child_weight", 1, 50, log=True),
                 subsample=trial.suggest_float("subsample", 0.6, 1.0),
                 colsample_bytree=trial.suggest_float("colsample_bytree", 0.4, 1.0),
                 reg_lambda=trial.suggest_float("reg_lambda", 1e-3, 20, log=True),
                 reg_alpha=trial.suggest_float("reg_alpha", 1e-3, 20, log=True))
    return run_cv(MODEL, p, Xs, ys, verbose=False)[0]

study = optuna.create_study(direction="maximize" if HIGHER_IS_BETTER else "minimize")
study.optimize(objective, n_trials=N_TRIALS, show_progress_bar=True)
best_params = study.best_params | ({"bootstrap_type": "Bernoulli"} if MODEL == "cat" else {})
print(study.best_value, best_params)

## CV complète avec les meilleurs paramètres, puis fit sur tout le train

In [ ]:
cv_score, oof, best_iter = run_cv(MODEL, best_params)

n_iter = int(best_iter * 1.1)   # un peu plus d'itérations car plus de données
final = make_model(MODEL, best_params | ({"iterations": n_iter, "early_stopping_rounds": None} if MODEL == "cat"
                                         else {"n_estimators": n_iter, "early_stopping_rounds": None}))
final.fit(X, y, **({} if MODEL == "cat" else {"verbose": False}))
test_pred = predict(final, X_test)

## Submission

In [ ]:
sub = sample.copy()
if IS_CLF and "auc" not in METRIC and "log" not in METRIC:   # prédiction de classe
    lab = test_pred.argmax(1) if test_pred.ndim > 1 else (test_pred > .5).astype(int)
    sub[TARGET] = classes[lab] if "classes" in globals() else lab
else:
    sub[TARGET] = test_pred
name = f"{MODEL}_cv{cv_score:.5f}.csv"
sub.to_csv(SUB_DIR / name, index=False)
print(f"submissions/{name}")
print(f'kg submit {{SLUG}} submissions/{name} -m "{MODEL} optuna" --cv {cv_score:.5f}')
sub.head()